This notebook represents the Bronze validation and split stage for the Automated Data Guard batch pipeline.

Purpose of Notebook 2:
* Read the raw Bronze dataset landed from the ingestion notebook
* Standardize schema and prepare the dataset for validation
* Apply data quality rules using a core-vs-optional field strategy
* Split records into Silver and Quarantine outputs
* Produce DQ metrics for ADF orchestration, SQL audit logging, and alerting

Architecture position:
* Upstream: Kaggle API ingestion into ADLS Bronze
* Current stage: Bronze validation and split
* Downstream: Silver trusted data, Quarantine rejected data, DQ metrics, audit logging, Gold aggregation

Design principles:
* Batch-friendly for Azure Data Factory
* Interview-friendly and easy to explain
* Clear separation between mandatory rule failures and optional quality observations
* Output-oriented so later orchestration can branch on DQ results

Expected inputs:
* `bronze_path`: Bronze CSV location in ADLS
* `silver_path`: target location for clean Silver output
* `quarantine_path`: target location for rejected records
* `alert_threshold_pct`: failure threshold used by ADF or Logic Apps
* Secret-backed access to Azure storage if needed by runtime

Expected outputs:
* Standardized Bronze DataFrame
* Silver output placeholder
* Quarantine output placeholder
* DQ metrics placeholder with values such as:
  * `total_rows`
  * `passed_rows`
  * `failed_rows`
  * `failure_pct`
  * `run_status`

ADF integration expectation:
* ADF should pass the runtime paths and threshold into this notebook
* This notebook should return simple metrics that ADF can use for SQL logging and alert branching

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Notebook 2 libraries loaded.")

Notebook 2 libraries loaded.


In [0]:
storage_account_name = "stdqprojectmouni"
default_ingestion_date = "2026-05-24"
default_bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/kaggle/us_accidents/ingestion_date={default_ingestion_date}/US_Accidents_March23.csv"
default_silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/us_accidents/validated/ingestion_date={default_ingestion_date}"
default_quarantine_path = f"abfss://quarantine@{storage_account_name}.dfs.core.windows.net/us_accidents/rejected/ingestion_date={default_ingestion_date}"
default_alert_threshold_pct = 5.0
default_demo_bad_batch_enabled = False

def get_widget_value(widget_name, default_value):
    try:
        value = dbutils.widgets.get(widget_name)
        return value if value not in (None, "") else default_value
    except Exception:
        return default_value

def parse_bool(value, default=False):
    if value is None:
        return default
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes", "y"}:
        return True
    if normalized in {"false", "0", "no", "n"}:
        return False
    return default

def parse_float(value, default):
    try:
        return float(value)
    except Exception:
        return default

try:
    dbutils.widgets.text("bronze_path", default_bronze_path, "bronze_path")
    dbutils.widgets.text("silver_path", default_silver_path, "silver_path")
    dbutils.widgets.text("quarantine_path", default_quarantine_path, "quarantine_path")
    dbutils.widgets.text("alert_threshold_pct", str(default_alert_threshold_pct), "alert_threshold_pct")
    dbutils.widgets.text("demo_bad_batch_enabled", str(default_demo_bad_batch_enabled).lower(), "demo_bad_batch_enabled")
except Exception:
    pass

bronze_path = get_widget_value("bronze_path", default_bronze_path)
silver_path = get_widget_value("silver_path", default_silver_path)
quarantine_path = get_widget_value("quarantine_path", default_quarantine_path)
alert_threshold_pct = parse_float(
    get_widget_value("alert_threshold_pct", str(default_alert_threshold_pct)),
    default_alert_threshold_pct
)
demo_bad_batch_enabled = parse_bool(
    get_widget_value("demo_bad_batch_enabled", str(default_demo_bad_batch_enabled).lower()),
    default_demo_bad_batch_enabled
)

storage_access_ready = False
storage_access_message = "Storage configuration not attempted yet."
config_source = "ADF/Notebook widgets with manual defaults"

try:
    storage_account_key = dbutils.secrets.get(scope="dq-secret-scope", key="storage-account-key")
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        storage_account_key
    )
    storage_access_ready = True
    storage_access_message = "Storage access configured from secret scope."
except Exception as e:
    storage_access_message = (
        "Shared-key Spark configuration is not available on this compute. "
        "Use dq-cluster or another compute that supports ADLS shared-key access. "
        f"Root cause: {str(e)}"
    )

print("Notebook 2 runtime configuration set.")
print("Configuration source:", config_source)
print("Bronze path:", bronze_path)
print("Silver path:", silver_path)
print("Quarantine path:", quarantine_path)
print("Alert threshold (%):", alert_threshold_pct)
print("Demo mode enabled:", demo_bad_batch_enabled)
print("Storage ready:", storage_access_ready)
print("Storage status:", storage_access_message)

Notebook 2 runtime configuration set.
Configuration source: ADF/Notebook widgets with manual defaults
Bronze path: abfss://bronze@stdqprojectmouni.dfs.core.windows.net/kaggle/us_accidents/ingestion_date=2026-05-24/US_Accidents_March23.csv
Silver path: abfss://silver@stdqprojectmouni.dfs.core.windows.net/us_accidents/validated/ingestion_date=2026-05-24
Quarantine path: abfss://quarantine@stdqprojectmouni.dfs.core.windows.net/us_accidents/rejected/ingestion_date=2026-05-24
Alert threshold (%): 5.0
Demo mode enabled: False
Storage ready: True
Storage status: Storage access configured from secret scope.


In [0]:
bronze_df = None

if storage_access_ready:
    bronze_df = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(bronze_path)
    )
    print("Bronze input loaded successfully.")
    display(bronze_df.limit(5))
else:
    print("Bronze input read skipped.")
    print(storage_access_message)

Bronze input loaded successfully.


ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
A-1,Source2,3,2016-02-08T05:46:00Z,2016-02-08T11:00:00Z,39.865147,-84.058723,null,null,0.01,Right lane blocked due to accident on I-70 Eastbound at Exit 41 OH-235 State Route 4.,I-70 E,Dayton,Montgomery,OH,45424,US,US/Eastern,KFFO,2016-02-08T05:58:00Z,36.9,null,91.0,29.68,10.0,Calm,null,0.02,Light Rain,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Night
A-2,Source2,2,2016-02-08T06:07:59Z,2016-02-08T06:37:59Z,39.92805900000001,-82.831184,null,null,0.01,Accident on Brice Rd at Tussing Rd. Expect delays.,Brice Rd,Reynoldsburg,Franklin,OH,43068-3402,US,US/Eastern,KCMH,2016-02-08T05:51:00Z,37.9,null,100.0,29.65,10.0,Calm,null,0.0,Light Rain,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Day
A-3,Source2,2,2016-02-08T06:49:27Z,2016-02-08T07:19:27Z,39.063148,-84.032608,null,null,0.01,Accident on OH-32 State Route 32 Westbound at Dela Palma Rd. Expect delays.,State Route 32,Williamsburg,Clermont,OH,45176,US,US/Eastern,KI69,2016-02-08T06:56:00Z,36.0,33.3,100.0,29.67,10.0,SW,3.5,null,Overcast,false,false,false,false,false,false,false,false,false,false,false,true,false,Night,Night,Day,Day
A-4,Source2,3,2016-02-08T07:23:34Z,2016-02-08T07:53:34Z,39.747753,-84.20558199999998,null,null,0.01,Accident on I-75 Southbound at Exits 52 52B US-35. Expect delays.,I-75 S,Dayton,Montgomery,OH,45417,US,US/Eastern,KDAY,2016-02-08T07:38:00Z,35.1,31.0,96.0,29.64,9.0,SW,4.6,null,Mostly Cloudy,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Day,Day,Day
A-5,Source2,2,2016-02-08T07:39:07Z,2016-02-08T08:09:07Z,39.627781,-84.188354,null,null,0.01,Accident on McEwen Rd at OH-725 Miamisburg Centerville Rd. Expect delays.,Miamisburg Centerville Rd,Dayton,Montgomery,OH,45459,US,US/Eastern,KMGY,2016-02-08T07:53:00Z,36.0,33.3,89.0,29.65,6.0,SW,3.5,null,Mostly Cloudy,false,false,false,false,false,false,false,false,false,false,false,true,false,Day,Day,Day,Day


In [0]:
import re

boolean_columns = [
    "amenity", "bump", "crossing", "give_way", "junction", "no_exit",
    "railway", "roundabout", "station", "stop", "traffic_calming",
    "traffic_signal", "turning_loop"
]

numeric_columns = [
    "severity", "start_lat", "start_lng", "end_lat", "end_lng", "distance_mi",
    "temperature_f", "wind_chill_f", "humidity", "pressure_in", "visibility_mi",
    "wind_speed_mph", "precipitation_in"
]

def normalize_column_name(name):
    normalized = re.sub(r"[^0-9a-zA-Z]+", "_", name.strip()).strip("_").lower()
    return normalized or "unnamed_column"

standardized_df = None

if bronze_df is None:
    print("Schema standardization skipped because Bronze data is not loaded.")
else:
    normalized_columns = [normalize_column_name(column) for column in bronze_df.columns]
    standardized_df = bronze_df.toDF(*normalized_columns)

    available_columns = set(standardized_df.columns)
    dtypes_map = dict(standardized_df.dtypes)

    transformed_columns = []
    for column_name in standardized_df.columns:
        column_expr = F.col(column_name)

        if dtypes_map.get(column_name) == "string":
            column_expr = F.trim(column_expr)

        if column_name in {"start_time", "end_time", "weather_timestamp"}:
            column_expr = F.to_timestamp(column_expr)

        if column_name in numeric_columns:
            if column_name == "severity":
                column_expr = column_expr.cast(T.IntegerType())
            else:
                column_expr = column_expr.cast(T.DoubleType())

        if column_name in boolean_columns and column_name in available_columns:
            column_expr = (
                F.when(F.lower(column_expr) == "true", F.lit(True))
                 .when(F.lower(column_expr) == "false", F.lit(False))
                 .otherwise(F.lit(None).cast(T.BooleanType()))
            )

        transformed_columns.append(column_expr.alias(column_name))

    standardized_df = standardized_df.select(*transformed_columns)

    print("Schema standardization completed.")
    print("Standardized columns:", len(standardized_df.columns))
    display(standardized_df.limit(5))

Schema standardization completed.
Standardized columns: 46


id,source,severity,start_time,end_time,start_lat,start_lng,end_lat,end_lng,distance_mi,description,street,city,county,state,zipcode,country,timezone,airport_code,weather_timestamp,temperature_f,wind_chill_f,humidity,pressure_in,visibility_mi,wind_direction,wind_speed_mph,precipitation_in,weather_condition,amenity,bump,crossing,give_way,junction,no_exit,railway,roundabout,station,stop,traffic_calming,traffic_signal,turning_loop,sunrise_sunset,civil_twilight,nautical_twilight,astronomical_twilight
A-1,Source2,3,2016-02-08T05:46:00Z,2016-02-08T11:00:00Z,39.865147,-84.058723,null,null,0.01,Right lane blocked due to accident on I-70 Eastbound at Exit 41 OH-235 State Route 4.,I-70 E,Dayton,Montgomery,OH,45424,US,US/Eastern,KFFO,2016-02-08T05:58:00Z,36.9,null,91.0,29.68,10.0,Calm,null,0.02,Light Rain,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Night
A-2,Source2,2,2016-02-08T06:07:59Z,2016-02-08T06:37:59Z,39.92805900000001,-82.831184,null,null,0.01,Accident on Brice Rd at Tussing Rd. Expect delays.,Brice Rd,Reynoldsburg,Franklin,OH,43068-3402,US,US/Eastern,KCMH,2016-02-08T05:51:00Z,37.9,null,100.0,29.65,10.0,Calm,null,0.0,Light Rain,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Day
A-3,Source2,2,2016-02-08T06:49:27Z,2016-02-08T07:19:27Z,39.063148,-84.032608,null,null,0.01,Accident on OH-32 State Route 32 Westbound at Dela Palma Rd. Expect delays.,State Route 32,Williamsburg,Clermont,OH,45176,US,US/Eastern,KI69,2016-02-08T06:56:00Z,36.0,33.3,100.0,29.67,10.0,SW,3.5,null,Overcast,false,false,false,false,false,false,false,false,false,false,false,true,false,Night,Night,Day,Day
A-4,Source2,3,2016-02-08T07:23:34Z,2016-02-08T07:53:34Z,39.747753,-84.20558199999998,null,null,0.01,Accident on I-75 Southbound at Exits 52 52B US-35. Expect delays.,I-75 S,Dayton,Montgomery,OH,45417,US,US/Eastern,KDAY,2016-02-08T07:38:00Z,35.1,31.0,96.0,29.64,9.0,SW,4.6,null,Mostly Cloudy,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Day,Day,Day
A-5,Source2,2,2016-02-08T07:39:07Z,2016-02-08T08:09:07Z,39.627781,-84.188354,null,null,0.01,Accident on McEwen Rd at OH-725 Miamisburg Centerville Rd. Expect delays.,Miamisburg Centerville Rd,Dayton,Montgomery,OH,45459,US,US/Eastern,KMGY,2016-02-08T07:53:00Z,36.0,33.3,89.0,29.65,6.0,SW,3.5,null,Mostly Cloudy,false,false,false,false,false,false,false,false,false,false,false,true,false,Day,Day,Day,Day


DQ strategy for this notebook:

Core mandatory fields:
* `ID`
* `Severity`
* `Start_Time`
* `Start_Lat`
* `Start_Lng`
* `State`
* `Country`

Core rule philosophy:
* If a core field is missing or invalid, the row should typically move to Quarantine
* These fields are required to trust the accident record for analytics

Optional or informational fields:
* `End_Lat`
* `End_Lng`
* `Wind_Chill(F)`
* `Precipitation(in)`
* `Weather_Condition`
* other sparse descriptive columns

Optional rule philosophy:
* Missing optional fields should not automatically reject the row
* They should be monitored and summarized in DQ metrics where useful

In [0]:
demo_bad_batch_sample_size = 5000
demo_bad_batch_seed = 42

dq_input_df = None
dq_input_mode = "REAL_BRONZE"
if standardized_df is None:
    print("Messy test dataset step skipped because standardized Bronze data is not available.")
else:
    dq_input_df = standardized_df
    print("DQ input is currently set to the real Bronze dataset.")

    if demo_bad_batch_enabled:
        demo_source_df = (
            standardized_df
            .limit(demo_bad_batch_sample_size)
            .withColumn("demo_row_id", F.monotonically_increasing_id())
        )

        dq_input_mode = "SYNTHETIC_BAD_BATCH"
        dq_input_df = (
            demo_source_df
            .withColumn(
                "id",
                F.when(F.col("demo_row_id") % 10 == 0, F.lit(None).cast("string"))
                 .when(F.col("demo_row_id") % 10 == 8, F.lit("DUPLICATE_DEMO_ID"))
                 .otherwise(F.col("id"))
            )
            .withColumn("severity", F.when(F.col("demo_row_id") % 10 == 1, F.lit(9)).otherwise(F.col("severity")))
            .withColumn("start_time", F.when(F.col("demo_row_id") % 10 == 2, F.lit(None).cast("timestamp")).otherwise(F.col("start_time")))
            .withColumn("start_lat", F.when(F.col("demo_row_id") % 10 == 3, F.lit(999.0)).otherwise(F.col("start_lat")))
            .withColumn("start_lng", F.when(F.col("demo_row_id") % 10 == 4, F.lit(-999.0)).otherwise(F.col("start_lng")))
            .withColumn("state", F.when(F.col("demo_row_id") % 10 == 5, F.lit("ZZ")).otherwise(F.col("state")))
            .withColumn("country", F.when(F.col("demo_row_id") % 10 == 6, F.lit("CA")).otherwise(F.col("country")))
            .withColumn(
                "end_time",
                F.when(
                    (F.col("demo_row_id") % 10 == 7) & F.col("start_time").isNotNull(),
                    F.expr("start_time - INTERVAL 1 HOUR")
                ).otherwise(F.col("end_time"))
            )
            .withColumn(
                "demo_bad_record_reason",
                F.when(F.col("demo_row_id") % 10 == 0, F.lit("Missing ID"))
                 .when(F.col("demo_row_id") % 10 == 1, F.lit("Invalid severity"))
                 .when(F.col("demo_row_id") % 10 == 2, F.lit("Missing start_time"))
                 .when(F.col("demo_row_id") % 10 == 3, F.lit("Out-of-range latitude"))
                 .when(F.col("demo_row_id") % 10 == 4, F.lit("Out-of-range longitude"))
                 .when(F.col("demo_row_id") % 10 == 5, F.lit("Invalid state"))
                 .when(F.col("demo_row_id") % 10 == 6, F.lit("Non-US country"))
                 .when(F.col("demo_row_id") % 10 == 7, F.lit("start_time after end_time"))
                 .when(F.col("demo_row_id") % 10 == 8, F.lit("Duplicate ID"))
                 .otherwise(F.lit("Pass control row"))
            )
        )

        print("Synthetic bad batch created for Quarantine and alert demonstration.")
        print("Demo input mode:", dq_input_mode)
        print("Demo sample size:", demo_bad_batch_sample_size)
        display(
            dq_input_df.select(
                "id", "severity", "start_time", "end_time",
                "start_lat", "start_lng", "state", "country", "demo_bad_record_reason"
            ).limit(10)
        )
    else:
        print("Demo mode is OFF. Real Bronze dataset will be validated and naturally split into Silver and Quarantine.")

DQ input is currently set to the real Bronze dataset.
Demo mode is OFF. Real Bronze dataset will be validated and naturally split into Silver and Quarantine.


In [0]:
validated_df = None

valid_us_states = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "DC"
]

if dq_input_df is None:
    print("DQ validation skipped because DQ input data is not available.")
else:
    print("DQ validation input mode:", dq_input_mode)

    id_present_rule = F.col("id").isNotNull() & (F.length(F.trim(F.col("id"))) > 0)
    severity_rule = F.col("severity").isin([1, 2, 3, 4])
    start_time_rule = F.col("start_time").isNotNull()
    start_lat_rule = F.col("start_lat").isNotNull() & F.col("start_lat").between(-90.0, 90.0)
    start_lng_rule = F.col("start_lng").isNotNull() & F.col("start_lng").between(-180.0, 180.0)
    state_rule = F.col("state").isNotNull() & F.upper(F.col("state")).isin(valid_us_states)
    country_rule = F.col("country").isNotNull() & (F.upper(F.col("country")) == F.lit("US"))
    end_time_rule = (
        F.col("end_time").isNull() |
        (F.col("start_time").isNotNull() & (F.col("start_time") <= F.col("end_time")))
    )
    zipcode_rule = F.col("zipcode").isNotNull() & F.col("zipcode").rlike(r"^[0-9]{5}(-[0-9]{4})?$")
    street_rule = F.col("street").isNotNull() & (F.length(F.trim(F.col("street"))) > 0)
    city_rule = F.col("city").isNotNull() & (F.length(F.trim(F.col("city"))) > 0)
    county_rule = F.col("county").isNotNull() & (F.length(F.trim(F.col("county"))) > 0)
    timezone_rule = F.col("timezone").isNotNull() & (F.length(F.trim(F.col("timezone"))) > 0)
    humidity_rule = F.col("humidity").isNull() | F.col("humidity").between(0.0, 100.0)
    visibility_rule = F.col("visibility_mi").isNull() | F.col("visibility_mi").between(0.0, 100.0)
    pressure_rule = F.col("pressure_in").isNull() | F.col("pressure_in").between(20.0, 35.0)
    temperature_rule = F.col("temperature_f").isNull() | F.col("temperature_f").between(-80.0, 150.0)
    weather_timestamp_rule = (
        F.col("weather_timestamp").isNull() |
        F.col("start_time").isNull() |
        (
            (F.col("weather_timestamp") >= F.col("start_time") - F.expr("INTERVAL 24 HOURS")) &
            (F.col("weather_timestamp") <= F.col("start_time") + F.expr("INTERVAL 6 HOURS"))
        )
    )

    validation_base_df = (
        dq_input_df
        .withColumn("id_present_rule", F.coalesce(id_present_rule, F.lit(False)))
        .withColumn("severity_rule", F.coalesce(severity_rule, F.lit(False)))
        .withColumn("start_time_rule", F.coalesce(start_time_rule, F.lit(False)))
        .withColumn("start_lat_rule", F.coalesce(start_lat_rule, F.lit(False)))
        .withColumn("start_lng_rule", F.coalesce(start_lng_rule, F.lit(False)))
        .withColumn("state_rule", F.coalesce(state_rule, F.lit(False)))
        .withColumn("country_rule", F.coalesce(country_rule, F.lit(False)))
        .withColumn("end_time_rule", F.coalesce(end_time_rule, F.lit(False)))
        .withColumn("zipcode_rule", F.coalesce(zipcode_rule, F.lit(False)))
        .withColumn("street_rule", F.coalesce(street_rule, F.lit(False)))
        .withColumn("city_rule", F.coalesce(city_rule, F.lit(False)))
        .withColumn("county_rule", F.coalesce(county_rule, F.lit(False)))
        .withColumn("timezone_rule", F.coalesce(timezone_rule, F.lit(False)))
        .withColumn("humidity_rule", F.coalesce(humidity_rule, F.lit(False)))
        .withColumn("visibility_rule", F.coalesce(visibility_rule, F.lit(False)))
        .withColumn("pressure_rule", F.coalesce(pressure_rule, F.lit(False)))
        .withColumn("temperature_rule", F.coalesce(temperature_rule, F.lit(False)))
        .withColumn("weather_timestamp_rule", F.coalesce(weather_timestamp_rule, F.lit(False)))
    )

    duplicate_ids_df = (
        validation_base_df
        .filter(F.col("id_present_rule"))
        .groupBy("id")
        .count()
        .filter(F.col("count") > 1)
        .select("id")
        .withColumn("duplicate_id_rule", F.lit(False))
    )

    validation_with_duplicates_df = (
        validation_base_df
        .join(duplicate_ids_df, on="id", how="left")
        .withColumn(
            "duplicate_id_rule",
            F.when(F.col("duplicate_id_rule").isNull(), F.lit(True)).otherwise(F.col("duplicate_id_rule"))
        )
    )

    dq_reason = F.concat_ws("; ",
        F.when(~F.col("id_present_rule"), F.lit("ID is missing")),
        F.when(~F.col("duplicate_id_rule"), F.lit("ID is duplicated")),
        F.when(~F.col("severity_rule"), F.lit("Severity must be between 1 and 4")),
        F.when(~F.col("start_time_rule"), F.lit("Start_Time is invalid or missing")),
        F.when(~F.col("start_lat_rule"), F.lit("Start_Lat is outside valid range")),
        F.when(~F.col("start_lng_rule"), F.lit("Start_Lng is outside valid range")),
        F.when(~F.col("state_rule"), F.lit("State is not a valid US state code")),
        F.when(~F.col("country_rule"), F.lit("Country is not US")),
        F.when(~F.col("end_time_rule"), F.lit("Start_Time is later than End_Time")),
        F.when(~F.col("zipcode_rule"), F.lit("Zipcode is missing or not in valid US format")),
        F.when(~F.col("street_rule"), F.lit("Street is missing")),
        F.when(~F.col("city_rule"), F.lit("City is missing")),
        F.when(~F.col("county_rule"), F.lit("County is missing")),
        F.when(~F.col("timezone_rule"), F.lit("Timezone is missing")),
        F.when(~F.col("humidity_rule"), F.lit("Humidity is outside realistic range")),
        F.when(~F.col("visibility_rule"), F.lit("Visibility is outside realistic range")),
        F.when(~F.col("pressure_rule"), F.lit("Pressure is outside realistic range")),
        F.when(~F.col("temperature_rule"), F.lit("Temperature is outside realistic range")),
        F.when(~F.col("weather_timestamp_rule"), F.lit("Weather timestamp is too far from Start_Time"))
    )

    optional_quality_flag = (
        F.col("weather_condition").isNull() |
        F.col("description").isNull() |
        F.col("end_lat").isNull() |
        F.col("end_lng").isNull() |
        F.col("wind_chill_f").isNull() |
        F.col("precipitation_in").isNull()
    )

    validated_df = (
        validation_with_duplicates_df
        .withColumn("dq_reason", dq_reason)
        .withColumn(
            "core_rule_pass",
            F.col("id_present_rule") &
            F.col("duplicate_id_rule") &
            F.col("severity_rule") &
            F.col("start_time_rule") &
            F.col("start_lat_rule") &
            F.col("start_lng_rule") &
            F.col("state_rule") &
            F.col("country_rule") &
            F.col("end_time_rule") &
            F.col("zipcode_rule") &
            F.col("street_rule") &
            F.col("city_rule") &
            F.col("county_rule") &
            F.col("timezone_rule") &
            F.col("humidity_rule") &
            F.col("visibility_rule") &
            F.col("pressure_rule") &
            F.col("temperature_rule") &
            F.col("weather_timestamp_rule")
        )
        .withColumn("optional_quality_flag", optional_quality_flag)
    )

    print("DQ validation logic applied.")
    display(validated_df.limit(5))

DQ validation input mode: REAL_BRONZE
DQ validation logic applied.


id,source,severity,start_time,end_time,start_lat,start_lng,end_lat,end_lng,distance_mi,description,street,city,county,state,zipcode,country,timezone,airport_code,weather_timestamp,temperature_f,wind_chill_f,humidity,pressure_in,visibility_mi,wind_direction,wind_speed_mph,precipitation_in,weather_condition,amenity,bump,crossing,give_way,junction,no_exit,railway,roundabout,station,stop,traffic_calming,traffic_signal,turning_loop,sunrise_sunset,civil_twilight,nautical_twilight,astronomical_twilight,id_present_rule,severity_rule,start_time_rule,start_lat_rule,start_lng_rule,state_rule,country_rule,end_time_rule,zipcode_rule,street_rule,city_rule,county_rule,timezone_rule,humidity_rule,visibility_rule,pressure_rule,temperature_rule,weather_timestamp_rule,duplicate_id_rule,dq_reason,core_rule_pass,optional_quality_flag
A-3228144,Source2,2,2017-10-06T17:16:48Z,2017-10-06T17:46:32Z,39.982758000000004,-75.278938,null,null,0.0,Accident on Manoa Rd at Harrogate Rd.,Harrogate Rd,Wynnewood,Montgomery,PA,19096-3532,US,US/Eastern,KPHL,2017-10-06T16:54:00Z,80.1,null,56.0,30.05,10.0,SW,9.2,null,Mostly Cloudy,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-3228142,Source2,3,2017-10-06T17:08:59Z,2017-10-06T17:53:42Z,40.732044,-73.919022,null,null,0.0,Right lane blocked due to accident on I-495 - lower level Eastbound at 48th St.,48th St,Woodside,Queens,NY,11377,US,US/Eastern,KLGA,2017-10-06T16:51:00Z,77.0,null,64.0,30.04,10.0,South,11.5,null,Scattered Clouds,false,false,true,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-4567219,Source1,2,2022-10-15T18:22:01Z,2022-10-15T19:37:03Z,38.946886,-77.41336700000002,38.946533,-77.41342399999998,0.025,Incident on CENTREVILLE RD near CENTREVILLE RD Drive with caution.,Centreville Rd,Herndon,Fairfax,VA,20171,US,US/Eastern,KIAD,2022-10-15T18:52:00Z,68.0,68.0,45.0,29.59,10.0,S,5.0,0.0,Fair,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,false
A-5514709,Source1,2,2021-12-17T20:35:00Z,2021-12-17T21:55:24Z,33.88963580115371,-117.55333542767737,33.89434580115371,-117.55747542767736,0.403,Incident on I-15 NB near HIDDEN VALLEY PKY Drive with caution.,I-15,Corona,Riverside,CA,92879,US,US/Pacific,KAJO,2021-12-17T20:56:00Z,59.0,59.0,23.0,29.58,10.0,SE,12.0,0.0,Fair,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Night,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,false
A-6143759,Source1,2,2021-04-09T03:14:00Z,2021-04-09T15:39:55Z,33.47572,-117.130096,33.480271,-117.139747,0.639,Slow traffic on CA-79 from Margarita Rd (CA-79) to I-15 (CA-79) due to accident.,Temecula Pkwy,Temecula,Riverside,CA,92592,US,US/Pacific,KF70,2021-04-09T03:15:00Z,48.0,48.0,87.0,28.53,10.0,CALM,0.0,0.0,Fair,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Night,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,false


In [0]:
silver_df = None
quarantine_df = None
silver_write_path = silver_path
quarantine_write_path = quarantine_path

if validated_df is None:
    print("Silver and Quarantine split skipped because validated data is not available.")
else:
    silver_df = validated_df.filter(F.col("core_rule_pass") == True)
    quarantine_df = validated_df.filter(F.col("core_rule_pass") == False)

    if dq_input_mode == "SYNTHETIC_BAD_BATCH":
        silver_write_path = f"{silver_path}_demo"
        quarantine_write_path = f"{quarantine_path}_demo"

    silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_write_path)
    quarantine_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(quarantine_write_path)

    print("Silver and Quarantine split completed.")
    print("Silver output path:", silver_write_path)
    print("Quarantine output path:", quarantine_write_path)
    print("Silver rows preview:")
    display(silver_df.limit(5))
    print("Quarantine rows preview:")
    display(quarantine_df.limit(5))

Silver and Quarantine split completed.
Silver output path: abfss://silver@stdqprojectmouni.dfs.core.windows.net/us_accidents/validated/ingestion_date=2026-05-24
Quarantine output path: abfss://quarantine@stdqprojectmouni.dfs.core.windows.net/us_accidents/rejected/ingestion_date=2026-05-24
Silver rows preview:


id,source,severity,start_time,end_time,start_lat,start_lng,end_lat,end_lng,distance_mi,description,street,city,county,state,zipcode,country,timezone,airport_code,weather_timestamp,temperature_f,wind_chill_f,humidity,pressure_in,visibility_mi,wind_direction,wind_speed_mph,precipitation_in,weather_condition,amenity,bump,crossing,give_way,junction,no_exit,railway,roundabout,station,stop,traffic_calming,traffic_signal,turning_loop,sunrise_sunset,civil_twilight,nautical_twilight,astronomical_twilight,id_present_rule,severity_rule,start_time_rule,start_lat_rule,start_lng_rule,state_rule,country_rule,end_time_rule,zipcode_rule,street_rule,city_rule,county_rule,timezone_rule,humidity_rule,visibility_rule,pressure_rule,temperature_rule,weather_timestamp_rule,duplicate_id_rule,dq_reason,core_rule_pass,optional_quality_flag
A-480,Source2,2,2016-03-04T06:10:42Z,2016-03-04T06:40:42Z,39.943745,-83.073151,null,null,0.01,Accident on Hague Ave at Sullivant Ave. Expect delays.,Sullivant Ave,Columbus,Franklin,OH,43204-2413,US,US/Eastern,KTZR,2016-03-04T06:15:00Z,31.6,28.3,100.0,30.04,2.5,NNW,3.5,null,Light Snow,false,false,false,false,false,false,false,false,true,false,false,true,false,Night,Night,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-551,Source2,2,2016-03-09T06:50:34Z,2016-03-09T07:20:34Z,40.028641,-83.01509899999998,null,null,0.01,Accident on High St at Como Ave. Expect delays.,W Como Ave,Columbus,Franklin,OH,43202-1027,US,US/Eastern,KOSU,2016-03-09T06:53:00Z,55.0,null,64.0,30.05,10.0,South,8.1,null,Clear,false,false,true,false,false,false,false,false,true,false,false,true,false,Night,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-775,Source2,3,2016-06-21T13:27:20Z,2016-06-21T14:13:00Z,38.551689,-121.69951599999999,null,null,0.0,Very slow traffic and shoulder blocked on exit ramp due to accident on I-80 Eastbound at I-80 Exit 75 / Mace Blvd.,I-80 E,Davis,Yolo,CA,95618,US,US/Pacific,KEDU,2016-06-21T13:35:00Z,93.2,null,20.0,29.97,10.0,WNW,3.5,null,Clear,false,false,false,false,true,false,false,false,false,false,false,false,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-891,Source2,2,2016-06-22T12:31:22Z,2016-06-22T13:01:22Z,37.381865999999995,-121.963829,null,null,0.0,Accident on County Hwy-G4 Montague Expy Eastbound near US-101.,Bayshore Fwy S,Santa Clara,Santa Clara,CA,95054,US,US/Pacific,KSJC,2016-06-22T12:53:00Z,73.9,null,50.0,30.0,10.0,WNW,11.5,null,Clear,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-980,Source2,2,2016-06-23T08:59:12Z,2016-06-23T09:44:12Z,38.660831,-121.33728799999999,null,null,0.0,Accident on Garfield Ave at Madison Ave.,Madison Ave,Sacramento,Sacramento,CA,95841-3111,US,US/Pacific,KMCC,2016-06-23T08:55:00Z,68.0,null,46.0,30.03,10.0,South,10.4,null,Clear,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true


Quarantine rows preview:


id,source,severity,start_time,end_time,start_lat,start_lng,end_lat,end_lng,distance_mi,description,street,city,county,state,zipcode,country,timezone,airport_code,weather_timestamp,temperature_f,wind_chill_f,humidity,pressure_in,visibility_mi,wind_direction,wind_speed_mph,precipitation_in,weather_condition,amenity,bump,crossing,give_way,junction,no_exit,railway,roundabout,station,stop,traffic_calming,traffic_signal,turning_loop,sunrise_sunset,civil_twilight,nautical_twilight,astronomical_twilight,id_present_rule,severity_rule,start_time_rule,start_lat_rule,start_lng_rule,state_rule,country_rule,end_time_rule,zipcode_rule,street_rule,city_rule,county_rule,timezone_rule,humidity_rule,visibility_rule,pressure_rule,temperature_rule,weather_timestamp_rule,duplicate_id_rule,dq_reason,core_rule_pass,optional_quality_flag
A-63197,Source2,2,2017-01-17T00:00:15Z,2017-01-17T00:43:49Z,34.099369,-117.819221,null,null,0.01,#1 lane blocked due to accident and car fire on CA-57 Southbound at Exit 24A Covina Blvd.,CA-57 N,San Dimas,Los Angeles,CA,91773,US,US/Pacific,KPOC,2017-01-17T06:47:00Z,41.0,null,87.0,30.11,10.0,Calm,null,null,Clear,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Night,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,false,true,Weather timestamp is too far from Start_Time,false,true
A-316926,Source2,3,2017-01-28T00:42:08Z,2017-01-28T01:12:08Z,34.12785,-117.918365,null,null,0.01,Accident on I-210 Westbound at Exit 39 Vernon Ave.,Foothill Fwy W,Azusa,Los Angeles,CA,91702,US,US/Pacific,KEMT,2017-01-28T07:46:00Z,41.0,null,70.0,30.46,10.0,Calm,null,null,Clear,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Night,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,false,true,Weather timestamp is too far from Start_Time,false,true
A-319102,Source2,2,2017-02-08T01:43:40Z,2017-02-08T02:12:31Z,34.267025,-118.44094799999999,null,null,0.01,Accident on CA-118 Eastbound at Exits 44A 44B I-5.,CA-118 E,Pacoima,Los Angeles,CA,91331,US,US/Pacific,KWHP,2017-02-08T07:50:00Z,57.2,null,100.0,30.22,4.0,NE,4.6,null,Overcast,false,false,false,false,false,false,false,false,false,false,false,true,false,Night,Night,Night,Night,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,false,true,Weather timestamp is too far from Start_Time,false,true
A-57391,Source2,2,2016-12-18T05:19:20Z,2016-12-18T05:48:01Z,34.007008,-117.964394,null,null,0.01,Right hand shoulder blocked due to accident on Sr-60 at Exit 16 Hacienda Blvd.,S Hacienda Blvd,Hacienda Heights,Los Angeles,CA,91745,US,US/Pacific,KEMT,2016-12-18T16:45:00Z,60.8,null,23.0,30.17,10.0,Calm,null,null,Clear,false,false,false,false,false,false,false,false,false,false,false,false,false,Night,Night,Night,Night,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,false,true,Weather timestamp is too far from Start_Time,false,true
A-89452,Source2,2,2016-08-14T02:52:55Z,2016-08-14T03:52:55Z,34.014107,-118.159203,null,null,0.0,Accident on Atlantic Blvd at Union Pacific Ave.,Union Pacific Ave,Los Angeles,Los Angeles,CA,90022-4929,US,US/Pacific,KCQT,2016-08-14T11:47:00Z,86.0,null,51.0,29.86,10.0,WSW,3.5,null,Clear,false,false,false,false,false,false,false,false,false,false,false,true,false,Night,Night,Night,Night,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,false,true,Weather timestamp is too far from Start_Time,false,true


In [0]:
if validated_df is None:
    dq_metrics = {
        "input_mode": dq_input_mode,
        "total_rows": None,
        "passed_rows": None,
        "failed_rows": None,
        "failure_pct": None,
        "run_status": "SKIPPED_NO_INPUT_DATA",
        "alert_threshold_pct": alert_threshold_pct,
    }
else:
    total_rows = validated_df.count()
    passed_rows = validated_df.filter(F.col("core_rule_pass") == True).count()
    failed_rows = validated_df.filter(F.col("core_rule_pass") == False).count()
    failure_pct = round((failed_rows / total_rows) * 100, 2) if total_rows > 0 else 0.0

    dq_metrics = {
        "input_mode": dq_input_mode,
        "total_rows": total_rows,
        "passed_rows": passed_rows,
        "failed_rows": failed_rows,
        "failure_pct": failure_pct,
        "run_status": "ALERT" if failure_pct > alert_threshold_pct else "SUCCESS",
        "alert_threshold_pct": alert_threshold_pct,
    }

print("DQ metrics:")
print(dq_metrics)

DQ metrics:
{'input_mode': 'REAL_BRONZE', 'total_rows': 7728394, 'passed_rows': 7693711, 'failed_rows': 34683, 'failure_pct': 0.45, 'run_status': 'SUCCESS', 'alert_threshold_pct': 5.0}


In [0]:
if validated_df is None:
    print("Verification skipped because validated data is not available.")
else:
    print("=== Verification Summary ===")
    print("Current input mode:", dq_input_mode)

    rule_summary_df = validated_df.select(
        F.count("*").alias("total_rows"),
        F.sum(F.when(~F.col("core_rule_pass"), 1).otherwise(0)).alias("failed_rows"),
        F.sum(F.when(~F.col("id_present_rule"), 1).otherwise(0)).alias("id_missing_failures"),
        F.sum(F.when(~F.col("duplicate_id_rule"), 1).otherwise(0)).alias("duplicate_id_failures"),
        F.sum(F.when(~F.col("severity_rule"), 1).otherwise(0)).alias("severity_failures"),
        F.sum(F.when(~F.col("start_time_rule"), 1).otherwise(0)).alias("start_time_failures"),
        F.sum(F.when(~F.col("start_lat_rule"), 1).otherwise(0)).alias("start_lat_failures"),
        F.sum(F.when(~F.col("start_lng_rule"), 1).otherwise(0)).alias("start_lng_failures"),
        F.sum(F.when(~F.col("state_rule"), 1).otherwise(0)).alias("state_failures"),
        F.sum(F.when(~F.col("country_rule"), 1).otherwise(0)).alias("country_failures"),
        F.sum(F.when(~F.col("end_time_rule"), 1).otherwise(0)).alias("end_time_failures")
    )

    print("Rule-level failure counts:")
    display(rule_summary_df)

    print("Rejected row sample:")
    display(
        validated_df
        .filter(~F.col("core_rule_pass"))
        .select(
            "id", "severity", "start_time", "end_time",
            "start_lat", "start_lng", "state", "country",
            "dq_reason",
            *(["demo_bad_record_reason"] if "demo_bad_record_reason" in validated_df.columns else [])
        )
        .limit(10)
    )

=== Verification Summary ===
Current input mode: REAL_BRONZE
Rule-level failure counts:


total_rows,failed_rows,id_missing_failures,duplicate_id_failures,severity_failures,start_time_failures,start_lat_failures,start_lng_failures,state_failures,country_failures,end_time_failures
7728394,34683,0,0,0,0,0,0,0,0,0


Rejected row sample:


id,severity,start_time,end_time,start_lat,start_lng,state,country,dq_reason
A-451819,2,2017-05-17T07:30:55Z,2017-05-17T08:45:44Z,30.40367100000001,-97.638527,TX,US,Weather timestamp is too far from Start_Time
A-462761,2,2017-06-23T01:07:03Z,2017-06-23T02:26:54Z,34.032115999999995,-118.153549,CA,US,Weather timestamp is too far from Start_Time
A-555953,1,2022-07-25T04:06:25Z,2022-07-25T05:34:53Z,30.26819,-82.70018,FL,US,Zipcode is missing or not in valid US format
A-576494,2,2022-07-03T16:59:40Z,2022-07-03T18:10:55Z,34.361301,-82.968102,GA,US,Zipcode is missing or not in valid US format
A-611379,3,2022-05-25T13:48:26Z,2022-05-25T14:16:53Z,26.171143,-80.856308,FL,US,Zipcode is missing or not in valid US format; Timezone is missing
A-613670,2,2022-05-23T12:39:10Z,2022-05-23T13:08:08Z,26.371677000000002,-80.169601,FL,US,Street is missing
A-630071,2,2022-05-03T12:28:35Z,2022-05-03T13:26:57Z,40.055217999999996,-83.032219,OH,US,Street is missing
A-712195,2,2022-02-13T10:40:00Z,2022-02-13T12:01:50Z,30.191544,-81.7043,FL,US,Weather timestamp is too far from Start_Time
A-63197,2,2017-01-17T00:00:15Z,2017-01-17T00:43:49Z,34.099369,-117.819221,CA,US,Weather timestamp is too far from Start_Time
A-316926,3,2017-01-28T00:42:08Z,2017-01-28T01:12:08Z,34.12785,-117.918365,CA,US,Weather timestamp is too far from Start_Time


Next steps for implementation:
* Move Azure SQL audit logging above the final notebook-exit payload so ADF receives audit status in the returned JSON
* Configure the SQL widgets and secrets, then run one full audit-enabled execution
* Run one controlled demo-mode execution to prove the alert-path behavior with `_demo` outputs
* Connect this notebook into the ADF parent pipeline after Notebook 1 and capture the final JSON return payload from the last cell

In [0]:
from pyspark.sql import functions as F

source_df = None
source_name = None

if "standardized_df" in globals() and standardized_df is not None:
    source_df = standardized_df
    source_name = "standardized_df"
elif "bronze_df" in globals() and bronze_df is not None:
    source_df = bronze_df
    source_name = "bronze_df"

if source_df is None:
    print("Bronze null-profile skipped because no in-memory Bronze dataframe is available in this session.")
    print("On the current serverless compute, direct ADLS access is failing because the storage account key configuration is invalid.")
    print("Run Cells 3-6 again on dq-cluster, then rerun this cell to profile nulls from the loaded dataframe.")
else:
    print(f"Profiling nulls from {source_name}.")

    column_options = {
        "end_lat_nulls": ["end_lat", "End_Lat"],
        "end_lng_nulls": ["end_lng", "End_Lng"],
        "wind_speed_nulls": ["wind_speed_mph", "Wind_Speed(mph)"],
        "precipitation_nulls": ["precipitation_in", "Precipitation(in)"],
        "city_nulls": ["city", "City"],
        "zipcode_nulls": ["zipcode", "Zipcode"],
        "weather_ts_nulls": ["weather_timestamp", "Weather_Timestamp"],
        "humidity_nulls": ["humidity", "Humidity(%)"],
        "visibility_nulls": ["visibility_mi", "Visibility(mi)"],
        "temperature_nulls": ["temperature_f", "Temperature(F)"]
    }

    available_columns = set(source_df.columns)
    aggregations = [F.count("*").alias("total_rows")]

    for alias_name, candidates in column_options.items():
        selected_column = next((column_name for column_name in candidates if column_name in available_columns), None)
        if selected_column is None:
            aggregations.append(F.lit(None).cast("long").alias(alias_name))
        else:
            aggregations.append(
                F.sum(F.col(selected_column).isNull().cast("int")).alias(alias_name)
            )

    profile = source_df.select(*aggregations)
    display(profile)

Profiling nulls from standardized_df.


total_rows,end_lat_nulls,end_lng_nulls,wind_speed_nulls,precipitation_nulls,city_nulls,zipcode_nulls,weather_ts_nulls,humidity_nulls,visibility_nulls,temperature_nulls
7728394,3402762,3402762,571233,2203586,253,1915,120228,174144,177098,163853


In [0]:
source_df = None
source_name = None

if "standardized_df" in globals() and standardized_df is not None:
    source_df = standardized_df
    source_name = "standardized_df"
elif "bronze_df" in globals() and bronze_df is not None:
    source_df = bronze_df
    source_name = "bronze_df"

if source_df is None:
    print("All-column null profile skipped because no in-memory Bronze dataframe is available in this session.")
    print("Run the Bronze read and standardization cells on dq-cluster, then rerun this cell.")
else:
    print(f"Computing null profile for every column from {source_name}.")

    total_rows = source_df.count()
    null_aggregations = [
        F.sum(F.col(column_name).isNull().cast("long")).alias(column_name)
        for column_name in source_df.columns
    ]

    null_counts_row = source_df.agg(*null_aggregations).collect()[0].asDict()

    null_profile_rows = [
        (
            column_name,
            int(null_counts_row[column_name]),
            round((null_counts_row[column_name] / total_rows) * 100, 4) if total_rows > 0 else 0.0
        )
        for column_name in source_df.columns
    ]

    null_profile_df = spark.createDataFrame(
        null_profile_rows,
        ["column_name", "null_count", "null_pct"]
    ).orderBy(F.col("null_count").desc(), F.col("column_name"))

    display(null_profile_df)

Computing null profile for every column from standardized_df.


column_name,null_count,null_pct
end_lat,3402762,44.0294
end_lng,3402762,44.0294
precipitation_in,2203586,28.5129
wind_chill_f,1999019,25.8659
wind_speed_mph,571233,7.3914
visibility_mi,177098,2.2915
wind_direction,175206,2.267
humidity,174144,2.2533
weather_condition,173459,2.2444
temperature_f,163853,2.1201


In [0]:
if validated_df is None:
    print("Full rule summary skipped because validated data is not available.")
else:
    full_rule_summary_df = validated_df.select(
        F.count("*").alias("total_rows"),
        F.sum(F.when(~F.col("core_rule_pass"), 1).otherwise(0)).alias("failed_rows"),
        F.sum(F.when(~F.col("id_present_rule"), 1).otherwise(0)).alias("id_present_failures"),
        F.sum(F.when(~F.col("duplicate_id_rule"), 1).otherwise(0)).alias("duplicate_id_failures"),
        F.sum(F.when(~F.col("severity_rule"), 1).otherwise(0)).alias("severity_failures"),
        F.sum(F.when(~F.col("start_time_rule"), 1).otherwise(0)).alias("start_time_failures"),
        F.sum(F.when(~F.col("start_lat_rule"), 1).otherwise(0)).alias("start_lat_failures"),
        F.sum(F.when(~F.col("start_lng_rule"), 1).otherwise(0)).alias("start_lng_failures"),
        F.sum(F.when(~F.col("state_rule"), 1).otherwise(0)).alias("state_failures"),
        F.sum(F.when(~F.col("country_rule"), 1).otherwise(0)).alias("country_failures"),
        F.sum(F.when(~F.col("end_time_rule"), 1).otherwise(0)).alias("end_time_failures"),
        F.sum(F.when(~F.col("zipcode_rule"), 1).otherwise(0)).alias("zipcode_failures"),
        F.sum(F.when(~F.col("street_rule"), 1).otherwise(0)).alias("street_failures"),
        F.sum(F.when(~F.col("city_rule"), 1).otherwise(0)).alias("city_failures"),
        F.sum(F.when(~F.col("county_rule"), 1).otherwise(0)).alias("county_failures"),
        F.sum(F.when(~F.col("timezone_rule"), 1).otherwise(0)).alias("timezone_failures"),
        F.sum(F.when(~F.col("humidity_rule"), 1).otherwise(0)).alias("humidity_failures"),
        F.sum(F.when(~F.col("visibility_rule"), 1).otherwise(0)).alias("visibility_failures"),
        F.sum(F.when(~F.col("pressure_rule"), 1).otherwise(0)).alias("pressure_failures"),
        F.sum(F.when(~F.col("temperature_rule"), 1).otherwise(0)).alias("temperature_failures"),
        F.sum(F.when(~F.col("weather_timestamp_rule"), 1).otherwise(0)).alias("weather_timestamp_failures")
    )

    display(full_rule_summary_df)

total_rows,failed_rows,id_present_failures,duplicate_id_failures,severity_failures,start_time_failures,start_lat_failures,start_lng_failures,state_failures,country_failures,end_time_failures,zipcode_failures,street_failures,city_failures,county_failures,timezone_failures,humidity_failures,visibility_failures,pressure_failures,temperature_failures,weather_timestamp_failures
7728394,34683,0,0,0,0,0,0,0,0,0,4932,10869,253,0,7808,0,15,448,31,12285


These are the rules that directly decide pass or fail.

| Rule | Field(s) used | Null allowed? | Clean condition | If it fails |
| --- | --- | --- | --- | --- |
| `id_present_rule` | `id` | No | `id` must exist and not be blank after trim | Row goes to Quarantine |
| `duplicate_id_rule` | `id` | Effectively no duplicate allowed | `id` must be unique among valid IDs | Row goes to Quarantine |
| `severity_rule` | `severity` | No | value must be `1, 2, 3, 4` | Row goes to Quarantine |
| `start_time_rule` | `start_time` | No | `start_time` must exist | Row goes to Quarantine |
| `start_lat_rule` | `start_lat` | No | must exist and be between `-90` and `90` | Row goes to Quarantine |
| `start_lng_rule` | `start_lng` | No | must exist and be between `-180` and `180` | Row goes to Quarantine |
| `state_rule` | `state` | No | must exist and be a valid 2-letter US state code | Row goes to Quarantine |
| `country_rule` | `country` | No | must exist and equal `US` | Row goes to Quarantine |
| `end_time_rule` | `end_time`, `start_time` | Yes for `end_time` | `end_time` can be null; if present, it must be `>= start_time` | Row goes to Quarantine |
| `zipcode_rule` | `zipcode` | No | must exist and match US ZIP format `12345` or `12345-6789` | Row goes to Quarantine |
| `street_rule` | `street` | No | must exist and not be blank | Row goes to Quarantine |
| `city_rule` | `city` | No | must exist and not be blank | Row goes to Quarantine |
| `county_rule` | `county` | No | must exist and not be blank | Row goes to Quarantine |
| `timezone_rule` | `timezone` | No | must exist and not be blank | Row goes to Quarantine |
| `humidity_rule` | `humidity` | Yes | can be null; if present, must be between `0` and `100` | Row goes to Quarantine |
| `visibility_rule` | `visibility_mi` | Yes | can be null; if present, must be between `0` and `100` | Row goes to Quarantine |
| `pressure_rule` | `pressure_in` | Yes | can be null; if present, must be between `20` and `35` | Row goes to Quarantine |
| `temperature_rule` | `temperature_f` | Yes | can be null; if present, must be between `-80` and `150` | Row goes to Quarantine |
| `weather_timestamp_rule` | `weather_timestamp`, `start_time` | Yes | can be null; if present, must be within `start_time - 24 hours` and `start_time + 6 hours` | Row goes to Quarantine |

Simple meaning of mandatory/core:

* these are the rules that define whether the accident record is trusted enough for Silver
* failing any one of them means the row is not clean enough

Optional/informational fields

These are not used in `core_rule_pass`. They do not directly reject the row. They are only tracked using `optional_quality_flag`.

| Field | Null allowed? | How used |
| --- | --- | --- |
| `weather_condition` | Yes | If null, `optional_quality_flag` becomes true |
| `description` | Yes | If null, `optional_quality_flag` becomes true |
| `end_lat` | Yes | If null, `optional_quality_flag` becomes true |
| `end_lng` | Yes | If null, `optional_quality_flag` becomes true |
| `wind_chill_f` | Yes | If null, `optional_quality_flag` becomes true |
| `precipitation_in` | Yes | If null, `optional_quality_flag` becomes true |

Simple meaning of optional:

* the row can still go to Silver even if these are missing
* these fields are useful, but not required to trust that the accident record is valid

Null vs non-null summary

Fields where null is not allowed

* `id`
* `severity`
* `start_time`
* `start_lat`
* `start_lng`
* `state`
* `country`
* `zipcode`
* `street`
* `city`
* `county`
* `timezone`

Fields where null is allowed

* `end_time`
* `humidity`
* `visibility_mi`
* `pressure_in`
* `temperature_f`
* `weather_timestamp`
* optional fields like `weather_condition`, `description`, `end_lat`, `end_lng`, `wind_chill_f`, `precipitation_in`

Why some nulls are allowed

* because missing enrichment is different from bad data
* example: an accident can still be real even if `humidity` is missing
* but an accident is not analytically trustworthy if `id` or `start_time` or `country` is missing

Clear rule conditions in plain language

* `id_present_rule`
  * Clean if ID exists and is not empty
  * Dirty if ID is null or blank

* `duplicate_id_rule`
  * Clean if the ID appears only once
  * Dirty if the same ID appears in multiple rows

* `severity_rule`
  * Clean if severity is `1` to `4`
  * Dirty if null or outside those values

* `start_time_rule`
  * Clean if start time exists
  * Dirty if null

* `start_lat_rule`
  * Clean if latitude exists and is within Earth range
  * Dirty if null or invalid

* `start_lng_rule`
  * Clean if longitude exists and is within Earth range
  * Dirty if null or invalid

* `state_rule`
  * Clean if state exists and is a valid US code like `CA`, `TX`, `NY`
  * Dirty if null or invalid like `ZZ`

* `country_rule`
  * Clean if country exists and is `US`
  * Dirty if null or not `US`

* `end_time_rule`
  * Clean if end time is null, or if it is after/equal to start time
  * Dirty only when end time exists but is before start time

* `zipcode_rule`
  * Clean if zipcode exists and matches US postal pattern
  * Dirty if null, blank, or badly formatted

* `street_rule`
  * Clean if street exists and is not blank
  * Dirty if null or blank

* `city_rule`
  * Clean if city exists and is not blank
  * Dirty if null or blank

* `county_rule`
  * Clean if county exists and is not blank
  * Dirty if null or blank

* `timezone_rule`
  * Clean if timezone exists and is not blank
  * Dirty if null or blank

* `humidity_rule`
  * Clean if humidity is null, or between `0` and `100`
  * Dirty only when present and outside range

* `visibility_rule`
  * Clean if visibility is null, or between `0` and `100`
  * Dirty only when present and outside range

* `pressure_rule`
  * Clean if pressure is null, or between `20` and `35`
  * Dirty only when present and outside range

* `temperature_rule`
  * Clean if temperature is null, or between `-80` and `150`
  * Dirty only when present and outside range

* `weather_timestamp_rule`
  * Clean if weather timestamp is null, or close enough to accident start time
  * Dirty only when present and too far away from `start_time`

Final simple classification

Mandatory identity and event fields

* `id`
* duplicate check on `id`
* `severity`
* `start_time`

Mandatory location fields

* `start_lat`
* `start_lng`
* `state`
* `country`
* `zipcode`
* `street`
* `city`
* `county`
* `timezone`

Conditional quality fields inside core rules

* `end_time`
* `humidity`
* `visibility_mi`
* `pressure_in`
* `temperature_f`
* `weather_timestamp`

Optional/informational fields

* `weather_condition`
* `description`
* `end_lat`
* `end_lng`
* `wind_chill_f`
* `precipitation_in`

In [0]:
import json

adf_payload = {
    "input_mode": dq_metrics.get("input_mode") if "dq_metrics" in globals() else None,
    "bronze_path": bronze_path if "bronze_path" in globals() else None,
    "silver_path": silver_write_path if "silver_write_path" in globals() else None,
    "quarantine_path": quarantine_write_path if "quarantine_write_path" in globals() else None,
    "total_rows": dq_metrics.get("total_rows") if "dq_metrics" in globals() else None,
    "passed_rows": dq_metrics.get("passed_rows") if "dq_metrics" in globals() else None,
    "failed_rows": dq_metrics.get("failed_rows") if "dq_metrics" in globals() else None,
    "failure_pct": dq_metrics.get("failure_pct") if "dq_metrics" in globals() else None,
    "run_status": dq_metrics.get("run_status") if "dq_metrics" in globals() else "SKIPPED_NO_METRICS",
    "alert_threshold_pct": dq_metrics.get("alert_threshold_pct") if "dq_metrics" in globals() else None,
    "storage_access_ready": storage_access_ready if "storage_access_ready" in globals() else None,
    "storage_access_message": storage_access_message if "storage_access_message" in globals() else None,
    "audit_logging_status": audit_logging_status if "audit_logging_status" in globals() else None,
    "audit_logging_message": audit_logging_message if "audit_logging_message" in globals() else None,
    "audit_log_target": audit_log_target if "audit_log_target" in globals() else None
}

adf_payload_json = json.dumps(adf_payload)

print("ADF payload preview before audit logging:")
print(adf_payload_json)
print("Final notebook exit payload is emitted after Azure SQL audit logging in the last cell.")

ADF payload preview before audit logging:
{"input_mode": "REAL_BRONZE", "bronze_path": "abfss://bronze@stdqprojectmouni.dfs.core.windows.net/kaggle/us_accidents/ingestion_date=2026-05-24/US_Accidents_March23.csv", "silver_path": "abfss://silver@stdqprojectmouni.dfs.core.windows.net/us_accidents/validated/ingestion_date=2026-05-24", "quarantine_path": "abfss://quarantine@stdqprojectmouni.dfs.core.windows.net/us_accidents/rejected/ingestion_date=2026-05-24", "total_rows": 7728394, "passed_rows": 7693711, "failed_rows": 34683, "failure_pct": 0.45, "run_status": "SUCCESS", "alert_threshold_pct": 5.0, "storage_access_ready": true, "storage_access_message": "Storage access configured from secret scope.", "audit_logging_status": null, "audit_logging_message": null, "audit_log_target": null}
Final notebook exit payload is emitted after Azure SQL audit logging in the last cell.


In [0]:
audit_logging_status = "NOT_ATTEMPTED"
audit_logging_message = "Audit logging not attempted yet."
audit_log_target = None

def get_secret_value(scope_name, secret_key):
    try:
        if not scope_name or not secret_key:
            return None
        return dbutils.secrets.get(scope=scope_name, key=secret_key)
    except Exception:
        return None

sql_server_name = get_widget_value("sql_server_name", "")
sql_database_name = get_widget_value("sql_database_name", "")
sql_table_name = get_widget_value("sql_table_name", "dq_run_audit")
sql_secret_scope = get_widget_value("sql_secret_scope", "")
sql_username_secret_key = get_widget_value("sql_username_secret_key", "")
sql_password_secret_key = get_widget_value("sql_password_secret_key", "")

try:
    dbutils.widgets.text("sql_server_name", "", "sql_server_name")
    dbutils.widgets.text("sql_database_name", "", "sql_database_name")
    dbutils.widgets.text("sql_table_name", "dq_run_audit", "sql_table_name")
    dbutils.widgets.text("sql_secret_scope", "", "sql_secret_scope")
    dbutils.widgets.text("sql_username_secret_key", "", "sql_username_secret_key")
    dbutils.widgets.text("sql_password_secret_key", "", "sql_password_secret_key")
except Exception:
    pass

sql_server_name = get_widget_value("sql_server_name", sql_server_name)
sql_database_name = get_widget_value("sql_database_name", sql_database_name)
sql_table_name = get_widget_value("sql_table_name", sql_table_name)
sql_secret_scope = get_widget_value("sql_secret_scope", sql_secret_scope)
sql_username_secret_key = get_widget_value("sql_username_secret_key", sql_username_secret_key)
sql_password_secret_key = get_widget_value("sql_password_secret_key", sql_password_secret_key)

sql_username = get_secret_value(sql_secret_scope, sql_username_secret_key)
sql_password = get_secret_value(sql_secret_scope, sql_password_secret_key)

required_sql_inputs_present = all([
    sql_server_name,
    sql_database_name,
    sql_table_name,
    sql_secret_scope,
    sql_username_secret_key,
    sql_password_secret_key,
    sql_username,
    sql_password,
])

if "dq_metrics" not in globals() or dq_metrics is None:
    audit_logging_status = "SKIPPED_NO_METRICS"
    audit_logging_message = "Azure SQL audit logging skipped because dq_metrics is not available."
elif not required_sql_inputs_present:
    audit_logging_status = "SKIPPED_MISSING_CONFIG"
    audit_logging_message = (
        "Azure SQL audit logging skipped because SQL connection widgets/secrets are not fully configured. "
        "Provide sql_server_name, sql_database_name, sql_table_name, sql_secret_scope, "
        "sql_username_secret_key, and sql_password_secret_key when ready."
    )
else:
    audit_log_target = f"{sql_server_name}/{sql_database_name}/{sql_table_name}"
    jdbc_url = (
        f"jdbc:sqlserver://{sql_server_name}:1433;"
        f"database={sql_database_name};"
        "encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
    )

    audit_record = [{
        "run_ts_utc": str(spark.sql("select current_timestamp() as ts").collect()[0]["ts"]),
        "input_mode": dq_metrics.get("input_mode"),
        "bronze_path": bronze_path,
        "silver_path": silver_write_path if "silver_write_path" in globals() else silver_path,
        "quarantine_path": quarantine_write_path if "quarantine_write_path" in globals() else quarantine_path,
        "total_rows": dq_metrics.get("total_rows"),
        "passed_rows": dq_metrics.get("passed_rows"),
        "failed_rows": dq_metrics.get("failed_rows"),
        "failure_pct": float(dq_metrics.get("failure_pct")) if dq_metrics.get("failure_pct") is not None else None,
        "run_status": dq_metrics.get("run_status"),
        "alert_threshold_pct": float(dq_metrics.get("alert_threshold_pct")) if dq_metrics.get("alert_threshold_pct") is not None else None,
        "storage_access_ready": bool(storage_access_ready) if "storage_access_ready" in globals() else None,
        "storage_access_message": storage_access_message if "storage_access_message" in globals() else None,
    }]

    audit_log_df = spark.createDataFrame(audit_record)

    try:
        (
            audit_log_df.write
            .format("jdbc")
            .mode("append")
            .option("url", jdbc_url)
            .option("dbtable", sql_table_name)
            .option("user", sql_username)
            .option("password", sql_password)
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .save()
        )
        audit_logging_status = "SUCCESS"
        audit_logging_message = f"Azure SQL audit row appended to {audit_log_target}."
    except Exception as e:
        audit_logging_status = "FAILED"
        audit_logging_message = f"Azure SQL audit logging failed: {str(e)}"

print("Audit logging status:", audit_logging_status)
print("Audit logging message:", audit_logging_message)
if audit_log_target:
    print("Audit log target:", audit_log_target)

final_adf_payload = {
    "input_mode": dq_metrics.get("input_mode") if "dq_metrics" in globals() else None,
    "bronze_path": bronze_path if "bronze_path" in globals() else None,
    "silver_path": silver_write_path if "silver_write_path" in globals() else None,
    "quarantine_path": quarantine_write_path if "quarantine_write_path" in globals() else None,
    "total_rows": dq_metrics.get("total_rows") if "dq_metrics" in globals() else None,
    "passed_rows": dq_metrics.get("passed_rows") if "dq_metrics" in globals() else None,
    "failed_rows": dq_metrics.get("failed_rows") if "dq_metrics" in globals() else None,
    "failure_pct": dq_metrics.get("failure_pct") if "dq_metrics" in globals() else None,
    "run_status": dq_metrics.get("run_status") if "dq_metrics" in globals() else "SKIPPED_NO_METRICS",
    "alert_threshold_pct": dq_metrics.get("alert_threshold_pct") if "dq_metrics" in globals() else None,
    "storage_access_ready": storage_access_ready if "storage_access_ready" in globals() else None,
    "storage_access_message": storage_access_message if "storage_access_message" in globals() else None,
    "audit_logging_status": audit_logging_status,
    "audit_logging_message": audit_logging_message,
    "audit_log_target": audit_log_target,
}

final_adf_payload_json = json.dumps(final_adf_payload)
print("Final ADF return payload:")
print(final_adf_payload_json)

try:
    dbutils.notebook.exit(final_adf_payload_json)
except Exception as e:
    print("Notebook exit skipped in interactive mode.")
    print(f"Exit reason: {str(e)}")